<a href="https://colab.research.google.com/github/csk01/lpg-cylinder-detection/blob/main/notebooks/lpg_detection_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LPG Cylinder Detection — YOLOv11 Training (v1)

## Overview
This notebook fine-tunes a **YOLOv11n** object detector on a Roboflow-hosted LPG cylinder
detection dataset (`lpg-identification-v2`, version 1). This is **stage 1** of the two-stage
pipeline: the trained detector finds and localizes the cylinder in a raw scene image, and its
crop is later fed into the brand classifier (stage 2). This is the v1 / nano baseline detector —
a larger `yolov11x` detector super­seded it later in the project (see the README's model version
history for `mAP50` comparisons).

## How to Run
1. **Runtime:** Colab **GPU** required in practice — training runs `model.train(..., device=0)`
   for 100 epochs, which is impractical on CPU. The notebook was last run on a Tesla T4 (see the
   `nvidia-smi` cell output).
2. **Credentials:** Requires a Roboflow API key. The `## Configure API keys` section below
   describes storing it as a Colab secret (`ROBOFLOW_API_KEY`), **but note the dataset-download
   cell currently hardcodes a literal API key string instead of reading the secret** — see the
   flagged comment there.
3. **Execution order:** Run cells top to bottom. Setup cells install dependencies and verify
   GPU access; the dataset cell downloads `lpg-identification-v2` version 1 in YOLOv11 format;
   the final cell trains `yolo11n.pt` for 100 epochs with the configured augmentation settings.
4. **Expected outputs:** a trained detector checkpoint (`best.pt` / `last.pt`) plus Ultralytics'
   standard training run artifacts (metrics, plots, sample batches) under
   `lpg-detection/v1-baseline/`.

YOLO11 builds on the advancements introduced in YOLOv9 and YOLOv10 earlier this year, incorporating improved architectural designs, enhanced feature extraction techniques, and optimized training methods.

YOLO11m achieves a higher mean mAP score on the COCO dataset while using 22% fewer parameters than YOLOv8m, making it computationally lighter without sacrificing performance.

YOLOv11 is available in 5 different sizes, ranging from `2.6M` to `56.9M` parameters, and capable of achieving from `39.5` to `54.7` mAP on the COCO dataset.

## Model / Dataset Info
| Component | Detail |
|---|---|
| Base checkpoint | `yolo11n.pt` (YOLOv11 nano, pretrained on COCO) |
| Dataset | Roboflow `lpg-identification-v2`, version 1, format `yolov11` |
| Task | Single-class(ish) cylinder detection (classes defined by the Roboflow project, not by this notebook) |
| Epochs | 100 |
| Image size | 640 |
| Batch size | 16 |
| Augmentations | `hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=15, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0, mixup=0.1` |
| Result metrics | Not printed/captured in this notebook's visible output — see Ultralytics run artifacts for `mAP50` etc. |

## Current Status
Training was executed at least once (dataset download and `nvidia-smi` cells show real,
non-empty outputs from a completed run). This is the **v1 / nano baseline** detector; per the
project's model history a `yolov11x` detector (v2, mAP50 ≈ 0.969) later superseded it as the
production detector. The dataset-download cell hardcodes a Roboflow API key directly in source
(flagged inline below) — this should be rotated/removed and replaced with the
`ROBOFLOW_API_KEY` secret pattern described in the Setup section.


## Setup



### Configure API keys

To fine-tune YOLO11, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install YOLO11 via Ultralytics

In [ ]:
!pip install roboflow ultralytics -q

## Fine-tune YOLO11 on custom dataset

**NOTE:** When training YOLOv11, make sure your data is located in `datasets`. If you'd like to change the default location of the data you want to use for fine-tuning, you can do so through Ultralytics' `settings.json`. In this tutorial, we will use one of the [datasets](https://universe.roboflow.com/liangdianzhong/-qvdww) available on [Roboflow Universe](https://universe.roboflow.com/). When downloading, make sure to select the `yolov11` export format.

In [ ]:
## Download dataset
from roboflow import Roboflow
# ⚠️ SECURITY: hardcoded Roboflow API key — should be rotated and replaced with
# userdata.get('ROBOFLOW_API_KEY') (the Colab secret set up in the "Configure API keys" step
# above) instead of a literal string committed to source control.
rf = Roboflow(api_key="dLVBQUrsHmhknSJJDGKc")
# Source detection dataset (images + bounding boxes) used to train the v1 nano detector
project = rf.workspace("krishnans-workspace-eu0zb").project("lpg-identification-v2")
version = project.version(1)
dataset = version.download("yolov11")

In [ ]:
from ultralytics import YOLO

# Start from the pretrained COCO nano checkpoint and fine-tune on the LPG cylinder dataset
model = YOLO("yolo11n.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # GPU device index (requires CUDA-enabled runtime)
    project="lpg-detection",
    name="v1-baseline",
    # Augmentation settings below add color jitter, rotation, translation, scaling, horizontal
    # flip, mosaic, and mixup to improve generalization on a relatively small custom dataset
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| `LPG-Identification-v2-1/` | `{HOME}/LPG-Identification-v2-1/` (Roboflow download location) | Downloaded YOLOv11-format dataset (images + label files + `data.yaml`) |
| `weights/best.pt` | `lpg-detection/v1-baseline/weights/best.pt` | Best-validation-mAP detector checkpoint from training — this is the v1 nano detector artifact |
| `weights/last.pt` | `lpg-detection/v1-baseline/weights/last.pt` | Final-epoch detector checkpoint |
| Training run artifacts (metrics, plots, sample batches, `results.csv`) | `lpg-detection/v1-baseline/` | Ultralytics' standard per-run training outputs |
